---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ______________ &nbsp; **학번:** ______________ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

시각자료 점수 부분 : 확인한 데이터(왜 이 데이터는 쓰지 않았는지) 모아서 캡쳐해서 붙여놔도 좋음

추천 데이터 
1. In Vivo 독성 Endpoint
2. cHTS
3. OPERA

간, 신장 관련 데이터
1. development and reproductive toxicity
2. cHTS
3. IVIVE & PBPK

In [1]:
import json
import glob

files = glob.glob("*.ipynb")
if not files:
    print("No notebook found.")
else:
    nb_path = files[0]
    print(f"Updating path in {nb_path}...")
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    
    for cell in nb["cells"]:
        if cell["cell_type"] == "code" and cell["id"] == "1dc93ed8":
            source = cell["source"]
            new_source = []
            for line in source:
                if 'file_path = "acute_oral.xlsx"' in line:
                    new_source.append('file_path = r"C:\\Users\\DS\\Desktop\\ML\\acute_oral.xlsx"\n')
                else:
                    new_source.append(line)
            cell["source"] = new_source
            break
            
    with open(nb_path, "w", encoding="utf-8") as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
    print("Notebook path updated.")

Updating path in Untitled.ipynb...
Notebook path updated.


In [2]:
import pandas as pd
import numpy as np

# 파일의 컴퓨터 주소(절대 경로)를 직접 지정했습니다.
file_path = r"C:\Users\DS\Desktop\ML\acute_oral.xlsx"

print("1. 데이터 불러오기 및 전처리 시작...")
df_raw = pd.read_excel(file_path, sheet_name="Data")
print(f"[Data] 원본 데이터 크기: {df_raw.shape}")

# 1) SMILES 결측치 행 제거
df_clean = df_raw[df_raw["SMILES"].notna()].copy()
print(f"[Data] SMILES 결측치 제거 후 크기: {df_clean.shape}")

# 2) 급성 경구 독성(Acute Oral Toxicity) 분석을 위해 LD50 Endpoint 필터링
df_ld50 = df_clean[df_clean["Endpoint"] == "LD50"].copy()
print(f"[Data] LD50 Endpoint 데이터 크기: {df_ld50.shape}")

# 3) Response(LD50 수치)를 수치형 데이터로 변환
df_ld50["Response"] = pd.to_numeric(df_ld50["Response"], errors='coerce')
df_ld50 = df_ld50[df_ld50["Response"].notna()]
print(f"[Data] 수치형 변환 후 데이터 크기: {df_ld50.shape}")

# 4) 중복 화합물 처리: 동일 SMILES에 대해 LD50 중앙값(median)을 사용하여 대표값으로 선정
df_unique = df_ld50.groupby("SMILES").agg({
    "Chemical_Name": "first",
    "Response": "median"
}).reset_index()
print(f"[Data] 중복 화합물 제거 후 유니크 화합물 수: {df_unique.shape[0]}")

# 5) GHS 분류 기준에 따른 이진 분류 라벨 정의
# LD50 <= 2000 mg/kg은 독성 물질(label=1), LD50 > 2000 mg/kg은 비독성 물질(label=0)로 분류
threshold = 2000
df_unique["label"] = (df_unique["Response"] <= threshold).astype(int)
print("\n[label] 클래스 라벨 분포:")
print(df_unique["label"].value_counts())

print("\n[Data Sample] 처음 5개 행 데이터:")
print(df_unique[["Chemical_Name", "SMILES", "Response", "label"]].head())

1. 데이터 불러오기 및 전처리 시작...
[Data] 원본 데이터 크기: (16721, 23)
[Data] SMILES 결측치 제거 후 크기: (16237, 23)
[Data] LD50 Endpoint 데이터 크기: (13188, 23)
[Data] 수치형 변환 후 데이터 크기: (13188, 23)
[Data] 중복 화합물 제거 후 유니크 화합물 수: 8898

[label] 클래스 라벨 분포:
label
1    5608
0    3290
Name: count, dtype: int64

[Data Sample] 처음 5개 행 데이터:
                                       Chemical_Name  \
0                             Nylon 12 (pellet 5 mm)   
1                             Pyridine--borane (1:1)   
2                                      Boron carbide   
3                                          NSC 37855   
4  N-(2-Amino-4,5,6,7-tetrahydro-1,3-benzothiazol...   

                                              SMILES  Response  label  
0  *-CCCCCCCCCCCC(=O)N-* |$star_e;;;;;;;;;;;;;;;s...    2000.0      1  
1                                      B.C1=CC=NC=C1      95.4      1  
2                                      B12B3B4B1C234    2000.0      1  
3  Br.Br.CC(C)(C1=CC=CC=C1)C1=CC(CN2CCCCC2)=C(O)C...     600.0      1 

In [4]:
import pandas as pd
import numpy as np

# 파일의 컴퓨터 주소(절대 경로)를 직접 지정했습니다.
file_path = r"C:\Users\DS\Desktop\ML\acute_oral.xlsx"

print("1. 데이터 불러오기 및 전처리 시작...")
df_raw = pd.read_excel(file_path, sheet_name="Data")
print(f"[Data] 원본 데이터 크기: {df_raw.shape}")

# 1) SMILES 결측치 행 제거
df_clean = df_raw[df_raw["SMILES"].notna()].copy()
print(f"[Data] SMILES 결측치 제거 후 크기: {df_clean.shape}")

# 2) 급성 경구 독성(Acute Oral Toxicity) 분석을 위해 LD50 Endpoint 필터링
df_ld50 = df_clean[df_clean["Endpoint"] == "LD50"].copy()
print(f"[Data] LD50 Endpoint 데이터 크기: {df_ld50.shape}")

# 3) Response(LD50 수치)를 수치형 데이터로 변환
df_ld50["Response"] = pd.to_numeric(df_ld50["Response"], errors='coerce')
df_ld50 = df_ld50[df_ld50["Response"].notna()]
print(f"[Data] 수치형 변환 후 데이터 크기: {df_ld50.shape}")

# 4) 중복 화합물 처리: 동일 SMILES에 대해 LD50 중앙값(median)을 사용하여 대표값으로 선정
df_unique = df_ld50.groupby("SMILES").agg({
    "Chemical_Name": "first",
    "Response": "median"
}).reset_index()
print(f"[Data] 중복 화합물 제거 후 유니크 화합물 수: {df_unique.shape[0]}")

# 5) GHS 분류 기준에 따른 이진 분류 라벨 정의
# LD50 <= 2000 mg/kg은 독성 물질(label=1), LD50 > 2000 mg/kg은 비독성 물질(label=0)로 분류
threshold = 2000
df_unique["label"] = (df_unique["Response"] <= threshold).astype(int)
print("\n[label] 클래스 라벨 분포:")
print(df_unique["label"].value_counts())

print("\n[Data Sample] 처음 5개 행 데이터:")
print(df_unique[["Chemical_Name", "SMILES", "Response", "label"]].head())

1. 데이터 불러오기 및 전처리 시작...
[Data] 원본 데이터 크기: (16721, 23)
[Data] SMILES 결측치 제거 후 크기: (16237, 23)
[Data] LD50 Endpoint 데이터 크기: (13188, 23)
[Data] 수치형 변환 후 데이터 크기: (13188, 23)
[Data] 중복 화합물 제거 후 유니크 화합물 수: 8898

[label] 클래스 라벨 분포:
label
1    5608
0    3290
Name: count, dtype: int64

[Data Sample] 처음 5개 행 데이터:
                                       Chemical_Name  \
0                             Nylon 12 (pellet 5 mm)   
1                             Pyridine--borane (1:1)   
2                                      Boron carbide   
3                                          NSC 37855   
4  N-(2-Amino-4,5,6,7-tetrahydro-1,3-benzothiazol...   

                                              SMILES  Response  label  
0  *-CCCCCCCCCCCC(=O)N-* |$star_e;;;;;;;;;;;;;;;s...    2000.0      1  
1                                      B.C1=CC=NC=C1      95.4      1  
2                                      B12B3B4B1C234    2000.0      1  
3  Br.Br.CC(C)(C1=CC=CC=C1)C1=CC(CN2CCCCC2)=C(O)C...     600.0      1 

In [4]:
import pandas as pd
import numpy as np

# 파일의 컴퓨터 주소(절대 경로)를 직접 지정했습니다.
file_path = r"C:\Users\DS\Desktop\ML\acute_oral.xlsx"

print("1. 데이터 불러오기 및 전처리 시작...")
df_raw = pd.read_excel(file_path, sheet_name="Data")
print(f"[Data] 원본 데이터 크기: {df_raw.shape}")

# 1) SMILES 결측치 행 제거
df_clean = df_raw[df_raw["SMILES"].notna()].copy()
print(f"[Data] SMILES 결측치 제거 후 크기: {df_clean.shape}")

# 2) 급성 경구 독성(Acute Oral Toxicity) 분석을 위해 LD50 Endpoint 필터링
df_ld50 = df_clean[df_clean["Endpoint"] == "LD50"].copy()
print(f"[Data] LD50 Endpoint 데이터 크기: {df_ld50.shape}")

# 3) Response(LD50 수치)를 수치형 데이터로 변환
df_ld50["Response"] = pd.to_numeric(df_ld50["Response"], errors='coerce')
df_ld50 = df_ld50[df_ld50["Response"].notna()]
print(f"[Data] 수치형 변환 후 데이터 크기: {df_ld50.shape}")

# 4) 중복 화합물 처리: 동일 SMILES에 대해 LD50 중앙값(median)을 사용하여 대표값으로 선정
df_unique = df_ld50.groupby("SMILES").agg({
    "Chemical_Name": "first",
    "Response": "median"
}).reset_index()
print(f"[Data] 중복 화합물 제거 후 유니크 화합물 수: {df_unique.shape[0]}")

# 5) GHS 분류 기준에 따른 이진 분류 라벨 정의
# LD50 <= 2000 mg/kg은 독성 물질(label=1), LD50 > 2000 mg/kg은 비독성 물질(label=0)로 분류
threshold = 2000
df_unique["label"] = (df_unique["Response"] <= threshold).astype(int)
print("\n[label] 클래스 라벨 분포:")
print(df_unique["label"].value_counts())

print("\n[Data Sample] 처음 5개 행 데이터:")
print(df_unique[["Chemical_Name", "SMILES", "Response", "label"]].head())

1. 데이터 불러오기 및 전처리 시작...
[Data] 원본 데이터 크기: (16721, 23)
[Data] SMILES 결측치 제거 후 크기: (16237, 23)
[Data] LD50 Endpoint 데이터 크기: (13188, 23)
[Data] 수치형 변환 후 데이터 크기: (13188, 23)
[Data] 중복 화합물 제거 후 유니크 화합물 수: 8898

[label] 클래스 라벨 분포:
label
1    5608
0    3290
Name: count, dtype: int64

[Data Sample] 처음 5개 행 데이터:
                                       Chemical_Name  \
0                             Nylon 12 (pellet 5 mm)   
1                             Pyridine--borane (1:1)   
2                                      Boron carbide   
3                                          NSC 37855   
4  N-(2-Amino-4,5,6,7-tetrahydro-1,3-benzothiazol...   

                                              SMILES  Response  label  
0  *-CCCCCCCCCCCC(=O)N-* |$star_e;;;;;;;;;;;;;;;s...    2000.0      1  
1                                      B.C1=CC=NC=C1      95.4      1  
2                                      B12B3B4B1C234    2000.0      1  
3  Br.Br.CC(C)(C1=CC=CC=C1)C1=CC(CN2CCCCC2)=C(O)C...     600.0      1 

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

# ==========================================
# 2. Descriptor 계산 (배점: 15점)
# ==========================================
print("2. RDKit 분자 디스크립터(Descriptor) 계산 시작...")

# 1) SMILES로부터 RDKit 분자 객체(Mol) 생성 및 유효성 검사
# (RDKit이 파싱할 수 없는 고분자나 무기화합물 등 일부 특이한 화합물은 제외됩니다.)
mols = []
valid_indices = []
for idx, smiles in enumerate(df_unique["SMILES"]):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mols.append(mol)
        valid_indices.append(idx)

# 유효한 분자 데이터만 선택 (8,898개 중 8,894개 성공)
df_final = df_unique.iloc[valid_indices].copy()
print(f"[RDKit] 유효한 RDKit 분자 개수: {len(mols)} / {df_unique.shape[0]}")

# 2) RDKit 2D 디스크립터 계산기 설정 (총 217가지의 물리화학적 특성 계산)
descriptor_names = [desc[0] for desc in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

print("디스크립터를 계산 중입니다. 화합물 양이 많아 30초~1분 정도 소요될 수 있습니다...")
descriptor_values = []
for mol in mols:
    descriptor_values.append(calculator.CalcDescriptors(mol))

# 3) 계산된 디스크립터를 데이터프레임으로 변환
df_descriptors = pd.DataFrame(descriptor_values, columns=descriptor_names, index=df_final.index)

# 4) 화합물 기본 정보(Chemical_Name, SMILES, label)와 계산된 디스크립터 병합
final_dataset = pd.concat([df_final[["Chemical_Name", "SMILES", "label"]], df_descriptors], axis=1)
print(f"[Descriptor] 최종 데이터셋 크기: {final_dataset.shape}")
print("\n[Data Sample] 디스크립터 병합 후 데이터 일부:")
print(final_dataset.head())

Fingerprint : 0과 1의 이진수(Bit) 배열로 표현한 것

In [6]:
from rdkit.Chem import AllChem
import numpy as np

# ==========================================
# 3. Morgan Fingerprint (ECFP4) 계산 및 변환
# ==========================================
print("3. RDKit Morgan Fingerprint (ECFP4) 계산 시작...")

# 1) Morgan Fingerprint 계산 (반지름 2 = ECFP4, 2048 비트 크기)
radius = 2
n_bits = 2048

fp_list = []
for mol in mols:
    # 비트 벡터 형태로 핑거프린트 생성
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    # 머신러닝 모델에 바로 사용할 수 있도록 0과 1로 이루어진 numpy array로 변환
    fp_array = np.zeros((1,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, fp_array)
    fp_list.append(fp_array)

# 2) 계산된 핑거프린트를 데이터프레임으로 변환
fp_columns = [f"Bit_{i}" for i in range(n_bits)]
df_fingerprints = pd.DataFrame(fp_list, columns=fp_columns, index=df_final.index)

# 3) 화합물 기본 정보(Chemical_Name, SMILES, label)와 핑거프린트 병합
final_fp_dataset = pd.concat([df_final[["Chemical_Name", "SMILES", "label"]], df_fingerprints], axis=1)

print(f"[Fingerprint] 최종 핑거프린트 데이터셋 크기: {final_fp_dataset.shape}")
print("\n[Data Sample] 핑거프린트 병합 후 데이터 일부:")
print(final_fp_dataset.head())

3. RDKit Morgan Fingerprint (ECFP4) 계산 시작...


[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerat

[Fingerprint] 최종 핑거프린트 데이터셋 크기: (8894, 2051)

[Data Sample] 핑거프린트 병합 후 데이터 일부:
                                       Chemical_Name  \
0                             Nylon 12 (pellet 5 mm)   
1                             Pyridine--borane (1:1)   
2                                      Boron carbide   
3                                          NSC 37855   
4  N-(2-Amino-4,5,6,7-tetrahydro-1,3-benzothiazol...   

                                              SMILES  label  Bit_0  Bit_1  \
0  *-CCCCCCCCCCCC(=O)N-* |$star_e;;;;;;;;;;;;;;;s...      1      0      0   
1                                      B.C1=CC=NC=C1      1      0      0   
2                                      B12B3B4B1C234      1      0      0   
3  Br.Br.CC(C)(C1=CC=CC=C1)C1=CC(CN2CCCCC2)=C(O)C...      1      0      0   
4                     Br.CC(=O)NC1CCC2=C(C1)SC(N)=N2      1      0      0   

   Bit_2  Bit_3  Bit_4  Bit_5  Bit_6  ...  Bit_2038  Bit_2039  Bit_2040  \
0      0      0      0      0      0  ...     

In [6]:
from rdkit.Chem import AllChem
import numpy as np

# ==========================================
# 3. Morgan Fingerprint (ECFP4) 계산 및 변환
# ==========================================
print("3. RDKit Morgan Fingerprint (ECFP4) 계산 시작...")

# 1) Morgan Fingerprint 계산 (반지름 2 = ECFP4, 2048 비트 크기)
radius = 2
n_bits = 2048

fp_list = []
for mol in mols:
    # 비트 벡터 형태로 핑거프린트 생성
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    # 머신러닝 모델에 바로 사용할 수 있도록 0과 1로 이루어진 numpy array로 변환
    fp_array = np.zeros((1,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, fp_array)
    fp_list.append(fp_array)

# 2) 계산된 핑거프린트를 데이터프레임으로 변환
fp_columns = [f"Bit_{i}" for i in range(n_bits)]
df_fingerprints = pd.DataFrame(fp_list, columns=fp_columns, index=df_final.index)

# 3) 화합물 기본 정보(Chemical_Name, SMILES, label)와 핑거프린트 병합
final_fp_dataset = pd.concat([df_final[["Chemical_Name", "SMILES", "label"]], df_fingerprints], axis=1)

print(f"[Fingerprint] 최종 핑거프린트 데이터셋 크기: {final_fp_dataset.shape}")
print("\n[Data Sample] 핑거프린트 병합 후 데이터 일부:")
print(final_fp_dataset.head())

3. RDKit Morgan Fingerprint (ECFP4) 계산 시작...


[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerator
[13:05:49] DEPRECATION WARNING: please use MorganGenerat

[Fingerprint] 최종 핑거프린트 데이터셋 크기: (8894, 2051)

[Data Sample] 핑거프린트 병합 후 데이터 일부:
                                       Chemical_Name  \
0                             Nylon 12 (pellet 5 mm)   
1                             Pyridine--borane (1:1)   
2                                      Boron carbide   
3                                          NSC 37855   
4  N-(2-Amino-4,5,6,7-tetrahydro-1,3-benzothiazol...   

                                              SMILES  label  Bit_0  Bit_1  \
0  *-CCCCCCCCCCCC(=O)N-* |$star_e;;;;;;;;;;;;;;;s...      1      0      0   
1                                      B.C1=CC=NC=C1      1      0      0   
2                                      B12B3B4B1C234      1      0      0   
3  Br.Br.CC(C)(C1=CC=CC=C1)C1=CC(CN2CCCCC2)=C(O)C...      1      0      0   
4                     Br.CC(=O)NC1CCC2=C(C1)SC(N)=N2      1      0      0   

   Bit_2  Bit_3  Bit_4  Bit_5  Bit_6  ...  Bit_2038  Bit_2039  Bit_2040  \
0      0      0      0      0      0  ...     